# TabPFN: Análisis Profundo sobre Datos Tabulares

## Propósito
Ejecutamos **TabPFN (Tabular Prior-Data Fitted Network)** sobre los dos datasets
del caso de estudio, generando todas las métricas, tiempos y visualizaciones
necesarias para analizar su rendimiento en profundidad.

No incluimos modelos baseline acá — esos están en `notebook1_baseline.ipynb`.
La comparativa formal entre baselines y TabPFN se construye aparte.

## Datasets
| Dataset | Tipo | Muestras | Features | Target |
|---|---|---:|---:|---|
| **German Credit** | Binario (good/bad) | 1,000 | 9 mixtas | `Risk` |
| **Crop Recommendation** | Multiclase (22 cultivos) | 2,200 | 7 numéricas | `label` |

## Referencia
Hollmann, N., Müller, S., Purucker, L. et al. (2023).
_Accurate predictions on small data with a tabular foundation model._ Nature.
https://doi.org/10.1038/s41586-024-08328-6


In [ ]:
!pip install tabpfn


In [ ]:
import time
import kagglehub

import random
import numpy as np
import torch
import pandas as pd
import os, sys, shutil

import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, roc_curve,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)
from tabpfn import TabPFNClassifier

timing_results = {}


In [ ]:
def set_seed(seed=42):
    """Configura la semilla en todas las librerías para reproducibilidad."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Semilla global configurada en {seed}.")

set_seed(42)


In [ ]:
def descargar_y_guardar_dataset(kaggle_id, nombre_archivo_local):
    os.makedirs("datasets_respaldo", exist_ok=True)
    ruta_respaldo = os.path.join("datasets_respaldo", nombre_archivo_local)
    try:
        print(f"Descargando '{kaggle_id}' desde Kaggle...")
        path = kagglehub.dataset_download(kaggle_id)
        archivo_csv = [f for f in os.listdir(path) if f.endswith('.csv')][0]
        shutil.copy(os.path.join(path, archivo_csv), ruta_respaldo)
        return pd.read_csv(ruta_respaldo)
    except Exception as e:
        print(f"Error de descarga: {e}")
        print("Buscando respaldo local...")
        if os.path.exists(ruta_respaldo):
            print(f"Cargando desde: {ruta_respaldo}")
            return pd.read_csv(ruta_respaldo)
        raise FileNotFoundError(
            f"No se encontro {nombre_archivo_local}. "
            "Descargalo de Kaggle manualmente."
        )

print("--- DATASET: GERMAN CREDIT ---")
df_german = descargar_y_guardar_dataset(
    kaggle_id="kabure/german-credit-data-with-risk",
    nombre_archivo_local="german_credit_data.csv"
)
print(f"Dimensiones: {df_german.shape}")
print(f"Columnas: {list(df_german.columns)}")

print("\n--- DATASET: CROP RECOMMENDATION ---")
df_crop = descargar_y_guardar_dataset(
    kaggle_id="atharvaingle/crop-recommendation-dataset",
    nombre_archivo_local="crop_recommendation.csv"
)
print(f"Dimensiones: {df_crop.shape}")
print(f"Columnas: {list(df_crop.columns)}")


In [ ]:
print("=== RADIOGRAFIA: GERMAN CREDIT ===")
print(f"Dimensiones: {df_german.shape}")
print(f"\nTipos de columna:\n{df_german.dtypes}")
print(f"\nPrimeras 3 filas:")
print(df_german.head(3).to_string())


In [ ]:
print("Preprocesando German Credit...")

if 'Unnamed: 0' in df_german.columns:
    df_german = df_german.drop(columns=['Unnamed: 0'])

target_col = 'Risk'
print(f"Columna objetivo: '{target_col}'")

df_processed = df_german.copy()
df_processed = df_processed.dropna(subset=[target_col])
X = df_processed.drop(columns=[target_col])
y = df_processed[target_col]

le = LabelEncoder()
y = le.fit_transform(y)
print(f"Clases detectadas: {le.classes_} -> [0, 1]")

categorical_cols = X.select_dtypes(include=['object', 'category']).columns
numeric_cols = X.select_dtypes(include=['number']).columns
if len(categorical_cols) > 0:
    X[categorical_cols] = X[categorical_cols].fillna('Desconocido')
if len(numeric_cols) > 0:
    X[numeric_cols] = X[numeric_cols].fillna(X[numeric_cols].median())

# OrdinalEncoder para TabPFN (tolera valores no vistos en fit)
if len(categorical_cols) > 0:
    encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X[categorical_cols] = encoder.fit_transform(X[categorical_cols].astype(str))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print("Preprocesamiento completado.")
print(f"Dimensiones -> Train: {X_train.shape} | Test: {X_test.shape}")


In [ ]:
os.environ["TABPFN_TOKEN"] = "tabpfn_sk_iPZbyLLnlUZKB1kKSHFg2Qu62IHXdl_NdvX8bGbL6V8"
os.environ["TABPFN_ALLOW_CPU_LARGE_DATASET"] = "1"

print("Iniciando TabPFN para German Credit...")

tabpfn_model = TabPFNClassifier(ignore_pretraining_limits=True)

t0 = time.time()
tabpfn_model.fit(X_train, y_train)
t_fit_german = time.time() - t0
timing_results["tabpfn_german_fit"] = t_fit_german

t0 = time.time()
y_pred_german = tabpfn_model.predict(X_test)
y_prob_german = tabpfn_model.predict_proba(X_test)
t_pred_german = time.time() - t0
timing_results["tabpfn_german_pred"] = t_pred_german

print(f"TabPFN (German) - Fit: {t_fit_german:.2f}s | Predict: {t_pred_german:.4f}s")


In [ ]:
print("=== MÉTRICAS TABPFN (GERMAN CREDIT) ===")

accuracy = accuracy_score(y_test, y_pred_german)
f1 = f1_score(y_test, y_pred_german)
roc_auc = roc_auc_score(y_test, y_prob_german[:, 1])

print(f"{'Métrica':<20} {'Valor':<10}")
print('-' * 30)
print(f"{'Accuracy':<20} {accuracy:<10.4f}")
print(f"{'F1-Score':<20} {f1:<10.4f}")
print(f"{'ROC-AUC':<20} {roc_auc:<10.4f}")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
# --- Confusion Matrix ---
cm = confusion_matrix(y_test, y_pred_german)
ConfusionMatrixDisplay(cm, display_labels=le.classes_).plot(ax=axes[0,0], cmap='Greens', values_format='d')
axes[0,0].set_title('TabPFN: Matriz de Confusión')

# --- ROC Curve ---
fpr, tpr, _ = roc_curve(y_test, y_prob_german[:, 1])
axes[0,1].plot(fpr, tpr, color='darkgreen', lw=2, label=f'TabPFN (AUC = {roc_auc:.3f})')
axes[0,1].plot([0,1], [0,1], 'k--', alpha=0.5, label='Aleatorio')
axes[0,1].fill_between(fpr, tpr, alpha=0.15, color='green')
axes[0,1].set_xlabel('Tasa de Falsos Positivos (FPR)')
axes[0,1].set_ylabel('Tasa de Verdaderos Positivos (TPR)')
axes[0,1].set_title('TabPFN: Curva ROC')
axes[0,1].legend(loc='lower right')
axes[0,1].grid(True, alpha=0.3)

# --- Prob distribution by true class ---
for cls, color, label in zip([0, 1], ['#ff7f7f', '#7fbf7f'], ['Clase 0 (good)', 'Clase 1 (bad)']):
    mask = y_test == cls
    axes[1,0].hist(y_prob_german[mask, 1], bins=20, alpha=0.6, color=color, label=label)
axes[1,0].axvline(0.5, color='black', ls='--', alpha=0.4, label='Umbral 0.5')
axes[1,0].set_xlabel('Probabilidad predicha (clase 1)')
axes[1,0].set_ylabel('Frecuencia')
axes[1,0].set_title('TabPFN: Distribución de probabilidades por clase real')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)

# --- Timing bar ---
times = [t_fit_german, t_pred_german]
bars = axes[1,1].bar(['Fit', 'Predict'], times, color=['#2ecc71', '#27ae60'], width=0.5)
for bar, t in zip(bars, times):
    axes[1,1].text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                  f'{t:.3f}s', ha='center', va='bottom', fontsize=11)
axes[1,1].set_ylabel('Tiempo (s)')
axes[1,1].set_title('TabPFN: Tiempos de Ejecución')
axes[1,1].grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n--- TIEMPOS (German Credit) ---")
print(f"{'Fase':<15} {'Tiempo (s)':<12}")
print('-' * 27)
print(f"{'Fit':<15} {t_fit_german:<12.4f}")
print(f"{'Predict':<15} {t_pred_german:<12.4f}")
print(f"{'Total':<15} {t_fit_german + t_pred_german:<12.4f}")


In [ ]:
print("Preprocesando Crop Recommendation...")

if 'Unnamed: 0' in df_crop.columns:
    df_crop = df_crop.drop(columns=['Unnamed: 0'])

target_col = 'label'
print(f"Columna objetivo: '{target_col}'")

df_proc = df_crop.copy()
df_proc = df_proc.dropna(subset=[target_col])
X_c = df_proc.drop(columns=[target_col])
y_c = df_proc[target_col]

le_crop = LabelEncoder()
y_c = le_crop.fit_transform(y_c)
n_classes = len(le_crop.classes_)
print(f"Clases detectadas: {n_classes} cultivos")
print(f"Cultivos: {list(le_crop.classes_)}")

cat_cols = X_c.select_dtypes(include=['object', 'category']).columns
num_cols = X_c.select_dtypes(include=['number']).columns
if len(cat_cols) > 0:
    X_c[cat_cols] = X_c[cat_cols].fillna('Desconocido')
if len(num_cols) > 0:
    X_c[num_cols] = X_c[num_cols].fillna(X_c[num_cols].median())
if len(cat_cols) > 0:
    enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X_c[cat_cols] = enc.fit_transform(X_c[cat_cols].astype(str))

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_c, y_c, test_size=0.3, random_state=42, stratify=y_c
)
print("Preprocesamiento completado.")
print(f"Dimensiones -> Train: {X_train_c.shape} | Test: {X_test_c.shape}")


In [ ]:
print("Iniciando TabPFN para Crop Recommendation...")

tabpfn_crop = TabPFNClassifier()

t0 = time.time()
tabpfn_crop.fit(X_train_c, y_train_c)
t_fit_crop = time.time() - t0
timing_results["tabpfn_crop_fit"] = t_fit_crop

t0 = time.time()
y_pred_crop = tabpfn_crop.predict(X_test_c)
y_prob_crop = tabpfn_crop.predict_proba(X_test_c)
t_pred_crop = time.time() - t0
timing_results["tabpfn_crop_pred"] = t_pred_crop

print(f"TabPFN (Crop) - Fit: {t_fit_crop:.2f}s | Predict: {t_pred_crop:.4f}s")


In [ ]:
print("=== MÉTRICAS TABPFN (CROP RECOMMENDATION) ===")

acc = accuracy_score(y_test_c, y_pred_crop)
f1_macro = f1_score(y_test_c, y_pred_crop, average='macro')
roc_auc_ovr = roc_auc_score(y_test_c, y_prob_crop, multi_class='ovr', average='macro')

print(f"{'Métrica':<22} {'Valor':<10}")
print('-' * 32)
print(f"{'Accuracy':<22} {acc:<10.4f}")
print(f"{'F1-Score (Macro)':<22} {f1_macro:<10.4f}")
print(f"{'ROC-AUC (OVR Macro)':<22} {roc_auc_ovr:<10.6f}")

n_err = (y_pred_crop != y_test_c).sum()
n_total = len(y_test_c)
print(f"\nErrores: {n_err} / {n_total} ({100*n_err/n_total:.2f}%)")

fig, (ax_cm, ax_t) = plt.subplots(1, 2, figsize=(18, 8),
                                   gridspec_kw={"width_ratios": [2, 1]})
cm = confusion_matrix(y_test_c, y_pred_crop)
disp = ConfusionMatrixDisplay(cm, display_labels=le_crop.classes_)
disp.plot(ax=ax_cm, cmap='Greens', values_format='d', colorbar=False)
ax_cm.set_title("TabPFN: Matriz de Confusión (22 clases)", fontsize=13, fontweight="bold")
ax_cm.set_xticklabels(le_crop.classes_, rotation=90, fontsize=7)
ax_cm.set_yticklabels(le_crop.classes_, fontsize=7)

times_c = [t_fit_crop, t_pred_crop]
bars = ax_t.bar(["Fit", "Predict"], times_c, color=["#2ecc71", "#27ae60"], width=0.5)
for bar, t in zip(bars, times_c):
    ax_t.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
              f"{t:.3f}s", ha="center", va="bottom", fontsize=12)
ax_t.set_ylabel("Tiempo (s)")
ax_t.set_title("TabPFN: Tiempos de Ejecución", fontweight="bold")
ax_t.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Top-k accuracy (vectorizado)
sorted_idx = np.argsort(y_prob_crop, axis=1)
top2_acc = np.mean(np.any(sorted_idx[:, -2:] == y_test_c[:, None], axis=1))
top3_acc = np.mean(np.any(sorted_idx[:, -3:] == y_test_c[:, None], axis=1))

print(f"{'Top-2 Accuracy':<22} {top2_acc:<10.4f}")
print(f"{'Top-3 Accuracy':<22} {top3_acc:<10.4f}")

fig, (ax_tk, ax_roc) = plt.subplots(1, 2, figsize=(16, 6))

k_values = [1, 2, 3]
topk = [acc, top2_acc, top3_acc]
colors = ["#27ae60", "#2ecc71", "#82e0aa"]
bars = ax_tk.bar(k_values, topk, color=colors, width=0.5)
for bar, v in zip(bars, topk):
    ax_tk.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
              f"{v:.4f}", ha="center", va="bottom", fontsize=11)
ax_tk.set_xticks(k_values)
ax_tk.set_xticklabels([f"Top-{k}" for k in k_values])
ax_tk.set_ylim(0, 1.05)
ax_tk.set_ylabel("Accuracy")
ax_tk.set_title("TabPFN: Top-k Accuracy", fontweight="bold")
ax_tk.grid(True, axis="y", alpha=0.3)

err_mask = y_pred_crop != y_test_c
err_classes, err_counts = np.unique(y_test_c[err_mask], return_counts=True)
conflictive_idx = err_classes[np.argsort(err_counts)[-4:]] if len(err_classes) > 0 else np.arange(min(4, n_classes))

for i, cls in enumerate(conflictive_idx):
    y_bin = (y_test_c == cls).astype(int)
    if len(np.unique(y_bin)) < 2:
        continue
    fpr, tpr, _ = roc_curve(y_bin, y_prob_crop[:, cls])
    auc_cls = roc_auc_score(y_bin, y_prob_crop[:, cls])
    ax_roc.plot(fpr, tpr, lw=2, label=f"{le_crop.classes_[cls]} (AUC = {auc_cls:.3f})")
ax_roc.plot([0,1], [0,1], "k--", alpha=0.4)
ax_roc.set_xlabel("FPR")
ax_roc.set_ylabel("TPR")
ax_roc.set_title("TabPFN: Curvas ROC por clase (conflictivas)", fontweight="bold")
ax_roc.legend(fontsize=9)
ax_roc.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
print("\n--- ANÁLISIS DE ERRORES ---")
if n_err > 0:
    error_idx = np.where(err_mask)[0]
    error_df = pd.DataFrame({
        "Real": le_crop.classes_[y_test_c[error_idx]],
        "Predicho": le_crop.classes_[y_pred_crop[error_idx]],
        "Prob (real)": y_prob_crop[error_idx, y_test_c[error_idx]]
    })
    print(f"Probabilidad media asignada a la clase real en errores: {error_df['Prob (real)'].mean():.4f}")
    print("\nPares de confusión más frecuentes:")
    pairs = error_df.groupby(["Real", "Predicho"]).size().sort_values(ascending=False)
    for (real, pred), count in pairs.head(5).items():
        print(f"  {real} -> {pred}: {count} errores")

print("\n--- TIEMPOS (Crop Recommendation) ---")
print(f"{'Fase':<15} {'Tiempo (s)':<12}")
print("-" * 27)
print(f"{'Fit':<15} {t_fit_crop:<12.4f}")
print(f"{'Predict':<15} {t_pred_crop:<12.4f}")
print(f"{'Total':<15} {t_fit_crop + t_pred_crop:<12.4f}")


## Conclusiones — TabPFN

### German Credit
- TabPFN ofrece una alternativa directa sin preprocesamiento complejo ni tuning de hiperparámetros.
- El rendimiento es competitivo con los baselines (RF/GB) sin necesidad de búsqueda de hiperparámetros.

### Crop Recommendation
- Accuracy y AUC cercanos a 1.0: el dataset es altamente separable.
- Top-3 accuracy cercano a 1.0: cuando TabPFN se equivoca, la clase correcta está entre las 3 más probables.
- Las confusiones ocurren entre cultivos con requerimientos agronómicos similares.

### General
- TabPFN es particularmente valioso en datasets pequeños (German: 1000, Crop: 2200).
- No requiere tuning de hiperparámetros, lo que simplifica el flujo de trabajo.
- La comparativa formal con baselines se encuentra en `comparativa.py`.


In [ ]:
import json, os
import numpy as np
os.makedirs("resultados", exist_ok=True)

# ── Métricas ──
acc_g = accuracy_score(y_test, y_pred_german)
f1_g = f1_score(y_test, y_pred_german)
auc_g = roc_auc_score(y_test, y_prob_german[:, 1])

acc_c = accuracy_score(y_test_c, y_pred_crop)
f1_c = f1_score(y_test_c, y_pred_crop, average="macro")
auc_c = roc_auc_score(y_test_c, y_prob_crop, multi_class="ovr", average="macro")

metrics = {
    "German Credit": {
        "TabPFN": {
            "accuracy": float(acc_g), "f1": float(f1_g), "auc": float(auc_g),
            "fit_time": t_fit_german, "pred_time": t_pred_german
        },
    },
    "Crop": {
        "TabPFN": {
            "accuracy": float(acc_c), "f1": float(f1_c), "auc": float(auc_c),
            "fit_time": t_fit_crop, "pred_time": t_pred_crop
        },
    },
}

# Combinar con notebook1 si existe
if os.path.exists("resultados/metrics_notebook1.json"):
    with open("resultados/metrics_notebook1.json") as f:
        nb1 = json.load(f)
    for ds in nb1:
        if ds not in metrics:
            metrics[ds] = {}
        for model, m in nb1[ds].items():
            metrics[ds][model] = m

with open("resultados/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

# ── Predicciones ──
# Cargar predicciones de notebook1
preds_nb1 = {}
if os.path.exists("resultados/preds_notebook1.npz"):
    preds_nb1 = dict(np.load("resultados/preds_notebook1.npz"))

save_dict = dict(preds_nb1)
save_dict.update({
    "tabpfn_probs_g": np.array(y_prob_german, dtype=np.float64),
    "tabpfn_preds_g": np.array(y_pred_german, dtype=int),
    "tabpfn_probs_c": np.array(y_prob_crop, dtype=np.float64),
    "tabpfn_preds_c": np.array(y_pred_crop, dtype=object),
    "classes_c": le_crop.classes_,
})
# Si notebook1 no puso y_test, ponerlos
if "y_test_g" not in save_dict:
    save_dict["y_test_g"] = np.array(y_test, dtype=int)
if "y_test_c" not in save_dict:
    save_dict["y_test_c"] = np.array(y_test_c, dtype=object)

np.savez_compressed("resultados/preds.npz", **save_dict)
print("✅ Resultados combinados exportados a resultados/")

# Limpiar archivos temporales de notebook1
for f in ["metrics_notebook1.json", "preds_notebook1.npz"]:
    p = os.path.join("resultados", f)
    if os.path.exists(p):
        os.remove(p)
